### Imports

In [32]:
import pandas as pd
import json
import os

### Features

In [33]:
VARIABLES = {
    "V0001": "uf",
    "V0026": "situacao_domicilio",
    "A001": "tipo_domicilio",
    "A005012": "rede_agua",
    "A00601": "agua_canalizada",
    "A01901": "acesso_internet",

    "C006": "sexo",
    "C008": "idade",
    "C009": "raca",

    "D001": "alfabetizado",
    "D00901": "escolaridade",

    "I00102": "plano_saude",

    "VDF002": "renda_domiciliar",
    "VDF003": "renda_per_capita",

    "P00104": "peso",
    "W00103": "peso_final",
    "W00203": "altura",

    "P050": "fumante_atual",
    "P052": "fumou_passado",

    "P02801": "alcool_semana",
    "P035": "atividade_fisica",

    "Q00201": "hipertensao",
    "Q03001": "diabetes",
    "Q06306": "doenca_coracao",
    "Q068": "avc",
    "Q074": "asma",
    "Q079": "artrite",
    "Q088": "dort",
    "Q092": "depressao",
    "Q11605": "enfisema",
    "Q11606": "bronquite",
    "Q124": "insuficiencia_renal"
}


### Configuração do dicionário xls

In [34]:
arquivo_excel = "D:\Faculdade\TCCII\ibge-official-files\dicionario_PNS_microdados_2019.xls"
aba = "dicionário pns"

COL_POS = 0        # posição inicial
COL_SIZE = 1       # tamanho
COL_CODE = 2       # código da variável
COL_MAP_CODE = 5   # código da categoria
COL_MAP_VALUE = 6  # descrição da categoria


<>:1: SyntaxWarning: invalid escape sequence '\F'
<>:1: SyntaxWarning: invalid escape sequence '\F'
C:\Users\jvict\AppData\Local\Temp\ipykernel_36152\2528479043.py:1: SyntaxWarning: invalid escape sequence '\F'
  arquivo_excel = "D:\Faculdade\TCCII\ibge-official-files\dicionario_PNS_microdados_2019.xls"


### Leitura do Excel

In [35]:
df_meta = pd.read_excel(
    arquivo_excel,
    sheet_name=aba,
    header=None,
    skiprows=4,
)


### Normalização do código da variável (ffill)

In [36]:
df_meta[COL_CODE] = df_meta[COL_CODE].astype(str).str.strip()
df_meta[COL_CODE] = df_meta[COL_CODE].replace({"nan": None, "": None})
df_meta[COL_CODE] = df_meta[COL_CODE].ffill()


### Filtrar só as variáveis de interesse

In [37]:
df_meta = df_meta[df_meta[COL_CODE].isin(VARIABLES.keys())].copy()

if df_meta.empty:
    raise ValueError("Nenhuma variável encontrada no dicionário.")


### Gerar layout (colspecs)

In [38]:
# Converte posição e tamanho para numérico, forçando erro virar NaN
df_meta[COL_POS] = pd.to_numeric(df_meta[COL_POS], errors="coerce")
df_meta[COL_SIZE] = pd.to_numeric(df_meta[COL_SIZE], errors="coerce")

# Agora sim: mantém apenas linhas válidas de layout
df_layout = df_meta.dropna(subset=[COL_POS, COL_SIZE])


col_positions = []
col_names = []

for _, row in df_layout.iterrows():
    start = int(row[COL_POS]) - 1
    end = start + int(row[COL_SIZE])
    col_positions.append((start, end))
    col_names.append(row[COL_CODE])


### Gerar mapas de valores (categorias)

In [39]:
mapas_de_valores = {}

df_mapas = df_meta.dropna(subset=[COL_MAP_CODE, COL_MAP_VALUE])

for var, group in df_mapas.groupby(COL_CODE):
    group[COL_MAP_CODE] = group[COL_MAP_CODE].astype(str).str.replace(r"\.0$", "", regex=True)
    group[COL_MAP_VALUE] = group[COL_MAP_VALUE].astype(str)

    mapas_de_valores[var] = (
        group.set_index(COL_MAP_CODE)[COL_MAP_VALUE]
        .to_dict()
    )


In [40]:
config = {
    "col_positions": col_positions,
    "col_names": col_names,
    "rename_map": VARIABLES,
    "mapas_de_valores": mapas_de_valores
}

os.makedirs("data/config", exist_ok=True)

with open("data/config/pns_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4, ensure_ascii=False)
